# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a practical walkthrough for loading, exploring, and analyzing the [FAIR²](https://doi.org/10.71728/senscience.vq4a-28xa) mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL, allowing standardized, FAIR access.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the Croissant metadata URL for the FAIR² dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Instantiate the Dataset and load the Croissant metadata
dataset = mlc.Dataset(croissant_url)

# You can view the metadata as a Python dict
metadata_json = dataset.metadata.to_json()
print(f"Dataset: {dataset.metadata.name}\n\nDescription: {dataset.metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We'll list available record sets and their respective fields. All referencing is by `@id`, as required by the Croissant convention.

In [ ]:
# Examine all record sets and their fields by @id
if hasattr(dataset.metadata, 'record_sets'):
    print("Record Sets found in metadata:\n")
    for rs in dataset.metadata.record_sets:
        print(f"  RecordSet '@id': {rs.id}")
        print(f"    name        : {rs.name}")
        if hasattr(rs, 'fields') and rs.fields:
            print("    Fields:")
            for field in rs.fields:
                print(f"      - @id: {field.id} | name: {field.name}")
        print("")
else:
    print("No record sets available in this dataset metadata.")

## 3. Data Extraction
Load data from a specific record set into a pandas DataFrame for analysis. Use the record set and field `@id`s from the overview above.

Below we demonstrate how to extract all record sets and create a DataFrame for each. We print the columns for reference.

In [ ]:
dataframes = {}
record_set_ids = []
# Find all available record_set @ids
if hasattr(dataset.metadata, 'record_sets'):
    for rs in dataset.metadata.record_sets:
        record_set_ids.append(rs.id)

for record_set_id in record_set_ids:
    # Use the mlcroissant records() generator to extract records for each set
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns for Record Set '@id': {record_set_id}")
        print(df.columns.tolist())
        print('\nPreview:')
        display(df.head())
    else:
        print(f"[Warning] No records found for record set '@id': {record_set_id}\n")
if not dataframes:
    print("No record sets could be loaded into dataframes. Check dataset content.")

# For the rest of the notebook we'll select the first available record set with actual data.
if dataframes:
    selected_record_set_id = next(iter(dataframes.keys()))
    print(f"\nSelected primary Record Set for EDA: {selected_record_set_id}")
else:
    selected_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply basic EDA: filtering by a numeric field, normalizing, and optionally grouping by another field, all using Croissant `@id` references.

We will now select a numeric field (e.g. `coverage_percent` or `peptide_count`) and a group-by field if available.

In [ ]:
if selected_record_set_id:
    df = dataframes[selected_record_set_id]
    # Infer numeric fields (by dtype)
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_fields:
        # Use the first numeric field
        numeric_field = numeric_fields[0]
        print(f"Using numeric field for filtering and normalization: {numeric_field}\n")
        threshold = df[numeric_field].mean() if df[numeric_field].mean() > 0 else 10
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())
        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Find a possible group field (non-numeric, low cardinality)
        group_fields = [col for col in df.columns if pd.api.types.is_string_dtype(df[col]) and df[col].nunique() < max(20, int(0.2 * len(df)))]
        if group_fields:
            group_field = group_fields[0]
            print(f"\nGrouping by: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            display(grouped_df.head())
        else:
            group_field = None
            print("No suitable group field found for grouping.")
    else:
        print("This record set has no numeric fields to use for filtering or normalization.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib and seaborn. All field selections use Croissant `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id and not df.empty and numeric_fields:
    col_plot = numeric_fields[0]
    plt.figure(figsize=(8,5))
    sns.histplot(df[col_plot], kde=True)
    plt.title(f'Histogram of {col_plot}')
    plt.xlabel(col_plot)
    plt.ylabel('Count')
    plt.show()
    # If a group field is available, show a boxplot
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field, y=col_plot, data=df)
        plt.title(f'{col_plot} Distribution by {group_field}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Insufficient data for visualization.")

## 6. Conclusion
In this notebook, we've demonstrated how to explore and analyze the FAIR² mass spectrometry dataset programmatically using `mlcroissant` and Croissant `@id` conventions.

- **Loaded** dataset metadata and records from schema.
- **Enumerated** available record sets and fields by `@id`.
- **Extracted** data and performed basic EDA (filtering, normalization, grouping).
- **Visualized** distributions and relationships for selected numeric and group fields.

This workflow can be adapted for any Croissant-encoded FAIR dataset. For advanced analysis, consult the [mlcroissant documentation](https://mlcommons.github.io/croissant/latest/).

**Citation**: If you use this dataset, cite as: Kulka, M., Rodriguez Miera, S. and Marcet-Palacios, M. 2026. Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells. Frontiers. DOI: [10.71728/senscience.vq4a-28xa](https://doi.org/10.71728/senscience.vq4a-28xa)